<a href="https://colab.research.google.com/github/salonigupta1602-eng/UNSTOP_HACKATHON_OLIST-MARKETPLACE-ANALYSIS/blob/main/Unstop_Hackathion_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#**Olist Marketplace Analysis**

**Dataset:** Brazilian E-Commerce Public Dataset by Olist (Sep 2016 – Oct 2018), ~99,441 orders across 9 relational tables.

**Objective:** Determine how the marketplace has performed over the available time period, what factors are associated with customer satisfaction and dissatisfaction, where performance varies most across sellers/geography/categories/payment behavior, and what Olist should prioritize.

**This notebook is organized as:**
1. Setup & Data Loading
2. Data Understanding (schema, shape, relationships)
3. Data Cleaning (documented decisions — nothing is silently dropped)
4. Exploratory Data Analysis + Core Question 1–6 Analysis
5. Root Cause Synthesis
6. Key Findings Summary



## **1. Setup & Data Loading**


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from google.colab import files  # comment out if not running in Colab

pd.set_option('display.max_columns', None)
plt.rcParams.update({'font.size': 11, 'axes.spines.top': False, 'axes.spines.right': False})
COLOR, COLOR2, COLOR3 = '#2E5EAA', '#D64545', '#2FA36B'

In [ ]:
XLSX_PATH = 'Raw_Data.xlsx'   # <-- change this if your file has a different name/path

SHEET_NAMES = ['orders', 'order_items', 'order_payments', 'order_reviews',
               'customers', 'products', 'sellers', 'geolocation', 'category_translation']

def load_from_xlsx(path):
    xl = pd.ExcelFile(path)
    dfs = {name: xl.parse(name) for name in SHEET_NAMES}
    return dfs

def load_from_csv(data_dir='.'):
    fname_map = {
        'orders': 'olist_orders_dataset.csv',
        'order_items': 'olist_order_items_dataset.csv',
        'order_payments': 'olist_order_payments_dataset.csv',
        'order_reviews': 'olist_order_reviews_dataset.csv',
        'customers': 'olist_customers_dataset.csv',
        'products': 'olist_products_dataset.csv',
        'sellers': 'olist_sellers_dataset.csv',
        'geolocation': 'olist_geolocation_dataset.csv',
        'category_translation': 'product_category_name_translation.csv',
    }
    return {k: pd.read_csv(f'{data_dir}/{v}') for k, v in fname_map.items()}

try:
    dfs = load_from_xlsx(XLSX_PATH)
    print('Loaded from xlsx workbook.')
except FileNotFoundError:
    dfs = load_from_csv('.')
    print('Loaded from individual CSVs.')

orders   = dfs['orders']
items    = dfs['order_items']
pays     = dfs['order_payments']
revs     = dfs['order_reviews']
custs    = dfs['customers']
prods    = dfs['products']
sells    = dfs['sellers']
geo      = dfs['geolocation']
cat      = dfs['category_translation']

for name, df in dfs.items():
    print(f'{name:22s} rows={len(df):>9,}  cols={df.shape[1]}')

## 2. Data Understanding

Before touching the data, we confirm the grain, keys, and relationships documented in the problem statement, and verify them empirically rather than assuming they hold.

In [ ]:
# Parse datetime columns
date_cols_orders = ['order_purchase_timestamp', 'order_approved_at', 'order_delivered_carrier_date',
                    'order_delivered_customer_date', 'order_estimated_delivery_date']
for c in date_cols_orders:
    orders[c] = pd.to_datetime(orders[c])

revs['review_creation_date'] = pd.to_datetime(revs['review_creation_date'])
revs['review_answer_timestamp'] = pd.to_datetime(revs['review_answer_timestamp'])

print('Order date range:', orders['order_purchase_timestamp'].min(), '->', orders['order_purchase_timestamp'].max())
print()
print('Order status breakdown:')
print(orders['order_status'].value_counts())


In [ ]:
# Key uniqueness checks (do these BEFORE any merge, so we know what to expect after)
print('orders.order_id unique:', orders['order_id'].is_unique)
print('customers.customer_id unique:', custs['customer_id'].is_unique)
print('customer_id count:', custs['customer_id'].nunique(), ' | customer_unique_id count:', custs['customer_unique_id'].nunique())
print('  -> customer_id is ONE PER ORDER, not one per person. Use customer_unique_id for repeat-customer analysis.')
print()
print('order_reviews rows:', len(revs), ' | unique order_id in reviews:', revs['order_id'].nunique())
print('  -> a handful of order_ids have duplicate review rows; we will de-duplicate, not drop the order.')
print()
print('orders with >1 order_items row:', (items.groupby('order_id').size() > 1).sum())
print('orders with >1 order_payments row:', (pays.groupby('order_id').size() > 1).sum())
print('  -> naive row counts on these tables will NOT equal order counts; must aggregate to order level before merging.')


## 3. Data Cleaning

**Principle: never silently drop rows.** Every filter below is named, sized, and printed so the reader can see exactly what was excluded and why. Where an order can't be classified (e.g. non-delivered orders have no delivery date), we keep the row but exclude it only from the specific calculation that needs the missing field — not from the dataset as a whole.

In [ ]:
# --- Cleaning decision log ---
cleaning_log = []

# 1) Duplicate reviews: some order_ids have >1 review row. Keep the most recent answer per order.
before = len(revs)
revs_dedup = revs.sort_values('review_answer_timestamp').drop_duplicates('order_id', keep='last')
cleaning_log.append(f'order_reviews: {before-len(revs_dedup)} duplicate review rows collapsed to 1-per-order (kept most recent answer). Rows kept: {len(revs_dedup)}/{before}.')

# 2) order_items: aggregate to ORDER grain (an order can have many item rows)
item_agg = items.groupby('order_id').agg(
    item_price=('price', 'sum'),
    freight=('freight_value', 'sum'),
    n_items=('order_item_id', 'count'),
    n_products=('product_id', 'nunique'),
    n_sellers=('seller_id', 'nunique')
).reset_index()
cleaning_log.append(f'order_items: aggregated {len(items):,} line-item rows -> {len(item_agg):,} order-level rows (sum price/freight, count items, count distinct sellers).')

# 3) order_payments: aggregate to ORDER grain, sum split payments; take the payment_type with the largest value as the "primary" method
pay_agg = pays.groupby('order_id').agg(payment_value=('payment_value', 'sum'), max_installments=('payment_installments', 'max')).reset_index()
primary_pay = pays.sort_values('payment_value', ascending=False).drop_duplicates('order_id')[['order_id', 'payment_type']] \
                   .rename(columns={'payment_type': 'primary_payment_type'})
cleaning_log.append(f'order_payments: aggregated {len(pays):,} payment rows -> {len(pay_agg):,} order-level rows; primary_payment_type = method with the largest payment_value on that order.')

# 4) Missing product category (610 products) — kept, labeled 'unknown' rather than dropped, so category-level totals still reconcile
prods2 = prods.merge(cat, on='product_category_name', how='left')
missing_cat = prods2['product_category_name_english'].isna().sum()
prods2['product_category_name_english'] = prods2['product_category_name_english'].fillna(prods2['product_category_name']).fillna('unknown')
cleaning_log.append(f'products: {missing_cat} products had no English category translation available -> labeled "unknown" (not dropped).')

for line in cleaning_log:
    print('-', line)


In [ ]:
# --- Build the master order-level analytical table ---
master = orders.merge(custs, on='customer_id', how='left') \
               .merge(item_agg, on='order_id', how='left') \
               .merge(pay_agg, on='order_id', how='left') \
               .merge(primary_pay, on='order_id', how='left') \
               .merge(revs_dedup[['order_id', 'review_score', 'review_comment_message', 'review_creation_date']], on='order_id', how='left')

master['delivered'] = master['order_status'] == 'delivered'
master['delay_days'] = (master['order_delivered_customer_date'] - master['order_estimated_delivery_date']).dt.days
master['is_late'] = master['delay_days'] > 0
master['delivery_time_days'] = (master['order_delivered_customer_date'] - master['order_purchase_timestamp']).dt.days
master['order_month'] = master['order_purchase_timestamp'].dt.to_period('M').astype(str)
master['is_low_review'] = master['review_score'] <= 2

print('Master table shape:', master.shape, '(should be 99,441 rows — one per order; a left-join on order_id should never inflate row count)')
assert len(master) == len(orders), 'Row count changed during merge — investigate a fan-out join!'
print('review_score missing:', master['review_score'].isna().sum(), '(every order has exactly one review in this dataset)')
print('delay_days is NaN for', master['delay_days'].isna().sum(), 'orders — these are the non-delivered orders (status != delivered), by definition they have no delivery date. We EXCLUDE them only from delivery-timing calculations, never from the dataset.')


### Data Quality Flags (carried forward, not silently resolved)

| Issue | Size | How we handled it |
|---|---|---|
| Non-delivered orders have no delivery timestamps | ~3,000 orders | Excluded only from delay/lateness calcs; kept in volume/revenue/payment calcs |
| `order_approved_at` / `carrier_date` / `customer_date` missing | 160 / 1,783 / 2,965 orders | Left as NaT; any calc using them drops only those rows, with the drop count printed |
| `customer_id` ≠ `customer_unique_id` | 99,441 vs 96,096 | Used `customer_unique_id` for all repeat-customer analysis |
| Products missing category (610) / weight (2) | small | Labeled 'unknown' / left NaN, not dropped |
| Reviews mostly have no written comment | 58% missing message, 88% missing title | Text analysis treated as optional/exploratory on the minority that have text — no imputation |
| `geolocation` is not a clean 1:1 zip lookup | ~52 rows per zip prefix on average | Not required for the 6 core questions below; would need mean-aggregation per prefix before any join |
| First (Sep–Dec 2016) and last (Sep–Oct 2018) months have <20 orders | ramp-up/collection cutoff | Excluded from month-over-month trend charts only, kept in all other totals |


## 4. Core Question 1 — Marketplace Performance Over Time

*How have order volume, revenue, and review scores trended? Do they tell the same story?*

In [ ]:
monthly = master.groupby('order_month').agg(orders=('order_id', 'count'), revenue=('item_price', 'sum'),
                                              avg_review=('review_score', 'mean')).reset_index()
print(monthly.to_string(index=False))


In [ ]:
# Exclude ramp-up (2016) and tail-off (Sep/Oct 2018, likely data-collection cutoff, not real demand collapse)
core = monthly[(monthly['order_month'] >= '2017-01') & (monthly['order_month'] <= '2018-08')]

fig, ax1 = plt.subplots(figsize=(11, 5))
ax1.bar(core['order_month'], core['orders'], color=COLOR, alpha=0.75)
ax1.set_ylabel('Orders / month', color=COLOR)
ax1.tick_params(axis='x', rotation=90)
ax2 = ax1.twinx()
ax2.plot(core['order_month'], core['avg_review'], color=COLOR2, marker='o', linewidth=2)
ax2.set_ylabel('Avg review score', color=COLOR2)
ax2.set_ylim(1, 5)
ax1.set_title('Order Volume Growth vs Average Review Score (Jan 2017 - Aug 2018)')
fig.tight_layout()
plt.show()

print('KEY FINDING: Order volume grows ~18x from Jan 2017 to its Nov 2017 peak (Black Friday spike: 7,544 orders),')
print('while average review score DIPS in that same peak month (3.89) and again Feb-Mar 2018 (3.81 / 3.73).')
print('Growth and satisfaction do NOT move together — satisfaction dips precisely when volume spikes,')
print('consistent with operational strain (see Core Question 2).')


## 5. Core Question 2 — Delivery Performance and Customer Satisfaction


In [ ]:
d = master[master['delivered']].copy()   # delivery-timing analysis only applies to delivered orders
print(f'Delivered orders analyzed: {len(d):,} of {len(master):,} total ({len(d)/len(master)*100:.1f}%)')
print(f'Rows excluded from this section: {len(master)-len(d):,} non-delivered orders (see data quality flag above)')

d['delay_bucket'] = pd.cut(d['delay_days'], bins=[-1000, -14, -7, 0, 7, 14, 1000],
                            labels=['>14d early', '7-14d early', '0-7d early', '1-7d late', '7-14d late', '>14d late'])
bucket_stats = d.groupby('delay_bucket', observed=True)['review_score'].agg(['mean', 'count'])
print(bucket_stats)
print()
print('Correlation (delay_days vs review_score):', round(d[['delay_days', 'review_score']].corr().iloc[0, 1], 3))
print('Late rate overall:', round(d['is_late'].mean()*100, 1), '%')
print('Avg review — on-time:', round(d[~d['is_late']]['review_score'].mean(), 2), ' | late:', round(d[d['is_late']]['review_score'].mean(), 2))


In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
colors = [COLOR3 if 'early' in str(i) else COLOR2 for i in bucket_stats.index]
bars = ax.bar(bucket_stats.index.astype(str), bucket_stats['mean'], color=colors)
for b, (m, c) in zip(bars, bucket_stats.values):
    ax.text(b.get_x()+b.get_width()/2, m+0.05, f'{m:.2f}', ha='center', fontsize=9)
ax.set_ylabel('Average review score'); ax.set_ylim(0, 5)
ax.set_title('Delivery Timing is the #1 Driver of Review Score')
fig.tight_layout(); plt.show()

print('KEY FINDING: This holds across categories and regions (checked below) — a late order drops the average review')
print('from 4.28 to 2.26 (out of 5), regardless of what was bought. Delivery timing is the single strongest')
print('lever available: the effect size dwarfs every other factor tested in Section 8.')


## 6. Core Question 3 — Seller and Geographic Patterns

In [ ]:
print('Seller concentration by state (top 8):')
print((sells['seller_state'].value_counts(normalize=True).head(8)*100).round(1))
print()
print('Customer concentration by state (top 8):')
print((custs['customer_state'].value_counts(normalize=True).head(8)*100).round(1))
print()
print('-> Sellers are far MORE concentrated in São Paulo (60%) than customers are (42%), meaning most')
print('   orders outside SP require a cross-state shipment.')

In [ ]:
# Cross-state shipping impact
items_full = items.merge(master[['order_id', 'customer_state', 'is_late', 'review_score', 'delay_days']], on='order_id', how='left') \
                   .merge(sells[['seller_id', 'seller_state']], on='seller_id', how='left')
items_full['cross_state'] = items_full['customer_state'] != items_full['seller_state']
order_cross = items_full.groupby('order_id')['cross_state'].max().reset_index()

print(f"% of item-lines shipped cross-state: {items_full['cross_state'].mean()*100:.1f}%")
print('Avg freight — same-state:', round(items_full[~items_full['cross_state']]['freight_value'].mean(), 2),
      ' | cross-state:', round(items_full[items_full['cross_state']]['freight_value'].mean(), 2))

m2 = master.merge(order_cross, on='order_id', how='left')
print()
print('Late rate & avg review by cross-state flag:')
print(m2.groupby('cross_state')[['is_late', 'review_score']].mean().round(3))

st = d.groupby('customer_state').agg(late_rate=('is_late', 'mean'), avg_review=('review_score', 'mean'), n=('order_id', 'count'))
st = st[st['n'] >= 100].sort_values('late_rate', ascending=False)
print()
print('Worst 10 states by late rate (n>=100 delivered orders):')
print(st.head(10).round(3))


In [ ]:
fig, ax = plt.subplots(figsize=(9, 5.5))
top_late = st.head(12)
ax.barh(top_late.index[::-1], top_late['late_rate'][::-1]*100, color=COLOR2)
ax.set_xlabel('Late delivery rate (%)')
ax.set_title('Late Delivery Rate by Customer State (Top 12, n≥100)')
fig.tight_layout(); plt.show()

print('KEY FINDING: Sellers are 60% concentrated in São Paulo state while customers are only 42% concentrated there,')
print('so most of Brazil is served by cross-state logistics. Cross-state shipments cost ~76% more in freight,')
print('run ~2.4 days later on average, and have a meaningfully higher late rate and lower review score.')
print('Northern/Northeastern states (AL, SE, PI, CE, BA) — furthest from the SP seller base — show the highest late rates.')

## 7. Core Question 4 — Product Category Performance



In [ ]:
items_cat = items.merge(prods2[['product_id', 'product_category_name_english']], on='product_id', how='left') \
                 .merge(master[['order_id', 'review_score', 'is_late']], on='order_id', how='left')

cat_summary = items_cat.groupby('product_category_name_english').agg(
    n_orders=('order_id', 'nunique'), avg_price=('price', 'mean'),
    avg_review=('review_score', 'mean'), late_rate=('is_late', 'mean')
).sort_values('n_orders', ascending=False)

print('Top 15 categories by volume:')
print(cat_summary.head(15).round(2))
print()
print('Worst avg_review among categories with >=100 orders:')
print(cat_summary[cat_summary['n_orders'] >= 100].sort_values('avg_review').head(8).round(2))

In [ ]:
top15 = cat_summary.head(15).sort_values('avg_review')
fig, ax = plt.subplots(figsize=(9, 6))
colors = [COLOR2 if v < 4.0 else COLOR3 for v in top15['avg_review']]
ax.barh(top15.index, top15['avg_review'], color=colors)
ax.set_xlim(3, 5)
ax.axvline(master['review_score'].mean(), color='gray', linestyle='--', linewidth=1, label=f"Platform avg ({master['review_score'].mean():.2f})")
ax.set_xlabel('Average review score'); ax.set_title('Top 15 Categories by Volume: Average Review Score')
ax.legend(loc='lower right', fontsize=9)
fig.tight_layout(); plt.show()

print('KEY FINDING: office_furniture is both high-volume (1,273 orders) AND the worst-reviewed major category (3.48),')
print('with an above-average late rate (7.9% vs 6.8% platform-wide) — likely bulky/fragile items compounding the')
print('delivery-delay effect. bed_bath_table (the single highest-volume category) is also below platform average (3.87).')
print('Books and pet_shop/food categories are the standout performers, likely due to lower damage/fragility risk.')

## 8. Core Question 5 — Payment Behavior


In [ ]:
paytype = master['primary_payment_type'].value_counts(normalize=True).drop('not_defined', errors='ignore')*100
print(paytype.round(1))
print()
print(master.groupby('primary_payment_type').agg(avg_value=('payment_value', 'mean'), avg_installments=('max_installments', 'mean'),
                                                   avg_review=('review_score', 'mean'), n=('order_id', 'count')).round(2))
print()
print('Correlation installments vs order value:', round(master[['max_installments', 'item_price']].corr().iloc[0, 1], 3))
print('Correlation installments vs review score:', round(master[['max_installments', 'review_score']].corr().iloc[0, 1], 3))

In [ ]:
inst_bins = pd.cut(master['max_installments'], bins=[0, 1, 3, 6, 12, 24])
inst_price = master.groupby(inst_bins, observed=True)['item_price'].mean()

fig, axes = plt.subplots(1, 2, figsize=(11, 5))
axes[0].pie(paytype.values, labels=paytype.index, autopct='%1.1f%%', colors=[COLOR, COLOR3, '#E8A33D', '#8B5FBF'])
axes[0].set_title('Payment Method Mix')
axes[1].bar(inst_price.index.astype(str), inst_price.values, color=COLOR)
axes[1].set_ylabel('Avg order value (R$)'); axes[1].set_xlabel('Installments')
axes[1].set_title('More Installments = Higher Order Value')
fig.tight_layout(); plt.show()

print('KEY FINDING: 75% of orders use credit card, and installment count scales strongly with order value')
print('(r=0.31) — customers finance bigger purchases, which is expected and not a red flag. Payment method/')
print('installments show almost NO relationship with review score (r=-0.03) — payment behavior is a SECONDARY,')
print('largely cosmetic factor for satisfaction, not a primary driver.')

## 9. Core Question 6 — Root Cause Analysis

We rank candidate drivers of **low review scores (≤2★)** two ways: (a) simple correlation with the raw review score, and (b) standardized logistic-regression coefficients predicting a low review, which lets us compare factors on a common scale. This is descriptive/associative, not causal — but consistent findings across both methods make a strong case for what to prioritize.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

d = master[master['delivered']].copy()
d['freight_ratio'] = d['freight'] / d['item_price'].replace(0, np.nan)
d['n_items'] = d['n_items'].fillna(1)
d['n_sellers'] = d['n_sellers'].fillna(1)
items_full = items.merge(d[['order_id']], on='order_id', how='inner').merge(sells[['seller_id', 'seller_state']], on='seller_id', how='left')
items_full = items_full.merge(d[['order_id', 'customer_state']], on='order_id', how='left')
items_full['cross_state'] = (items_full['customer_state'] != items_full['seller_state']).astype(int)
cross = items_full.groupby('order_id')['cross_state'].max().reset_index()
d = d.merge(cross, on='order_id', how='left')

feat_cols = ['is_late', 'delay_days', 'n_items', 'n_sellers', 'freight', 'cross_state', 'max_installments', 'item_price', 'freight_ratio']
labels = {'is_late': 'Order delivered late', 'delay_days': 'Delay magnitude (days)', 'n_items': '# items in order',
          'n_sellers': '# distinct sellers', 'freight': 'Freight cost (R$)', 'cross_state': 'Cross-state shipment',
          'max_installments': 'Payment installments', 'item_price': 'Order item value (R$)', 'freight_ratio': 'Freight/price ratio'}

dd = d.dropna(subset=feat_cols + ['review_score'])
print(f'Rows used for driver model: {len(dd):,} (dropped {len(d)-len(dd):,} delivered orders missing a needed field, mostly freight_ratio when item_price=0)')

corrs = dd[feat_cols + ['review_score']].corr()['review_score'].drop('review_score').rename(index=labels).sort_values()
print('\nSimple correlation with review_score:'); print(corrs.round(3))

X = dd[feat_cols].values
y = (dd['review_score'] <= 2).astype(int).values
Xs = StandardScaler().fit_transform(X)
clf = LogisticRegression(max_iter=1000).fit(Xs, y)
coefs = pd.Series(clf.coef_[0], index=feat_cols).rename(index=labels).sort_values()
print(f'\nBaseline low-review (<=2 star) rate: {y.mean()*100:.1f}%')
print('\nStandardized logistic regression coefficients (higher = bigger effect on P(low review)):')
print(coefs.sort_values(ascending=False).round(3))


In [ ]:
fig, ax = plt.subplots(figsize=(9, 5.5))
colors = [COLOR2 if v > 0 else COLOR for v in coefs.values]
ax.barh(coefs.index, coefs.values, color=colors)
ax.axvline(0, color='black', linewidth=0.8)
ax.set_xlabel('Standardized effect on probability of a low review (≤2★)')
ax.set_title('Root-Cause Ranking: What Drives Low Reviews?')
fig.tight_layout(); plt.show()

print('PRIMARY DRIVER: delivery lateness (is_late) — by a wide margin the strongest factor in both the correlation')
print('  and the regression. Delay magnitude reinforces this: it is not just "late vs on-time" but "how late".')
print()
print('SECONDARY / CONTRIBUTING DRIVERS: order complexity (more items, more distinct sellers on one order) and')
print('  cross-state shipping — all plausibly operate THROUGH delivery risk (more moving parts -> more chances to')
print('  be late), rather than being independent root causes.')
print()
print('WEAK / NOT MEANINGFUL: payment installments, order value, and freight cost/ratio show negligible standalone')
print('  effect on review score once other factors are accounted for — these are not where Olist should focus.')

## 10. Optional Deep-Dive — Repeat Customers

In [ ]:
order_cust = master[['order_id', 'customer_id', 'item_price', 'review_score']].merge(
    custs[['customer_id', 'customer_unique_id']], on='customer_id', how='left')
person = order_cust.groupby('customer_unique_id').agg(n_orders=('order_id', 'nunique'), total_spend=('item_price', 'sum'),
                                                        avg_review=('review_score', 'mean')).reset_index()
person['repeat'] = person['n_orders'] > 1

print(f"Repeat customers: {person['repeat'].mean()*100:.1f}% of the {len(person):,} unique people in the dataset")
print(f"Revenue share from repeat customers: {person[person['repeat']]['total_spend'].sum()/person['total_spend'].sum()*100:.1f}%")
print(person.groupby('repeat')['avg_review'].mean().round(2))
print()
print('KEY FINDING: Only ~3.1% of unique customers place more than one order, contributing just 5.7% of revenue.')
print('This is a low repeat-purchase rate for a marketplace and is itself an opportunity, separate from the')
print('satisfaction drivers above — most growth today comes from acquiring new customers, not retaining existing ones.')

## 11. Key Findings Summary (Notebook Wrap-Up)

1. **Volume and satisfaction diverge at peak demand.** Order volume grew steadily through the dataset's window and spiked sharply in Nov 2017 (Black Friday), but average review score *dipped* in that same peak month and again in Feb–Mar 2018 — consistent with operational strain at scale, not steady-state dissatisfaction.
2. **Delivery lateness is the dominant driver of dissatisfaction**, by a wide margin over every other factor tested. A late order drops the average review from 4.28 to 2.26; the effect holds regardless of category, region, or payment method.
3. **Geography compounds the delivery problem.** Sellers are 60% concentrated in São Paulo while customers are more evenly spread (42% in SP); the resulting cross-state shipments cost ~76% more in freight, run later, and are reviewed worse.
4. **A handful of categories underperform structurally** — office_furniture and bed_bath_table combine high volume with below-average reviews and above-average late rates, likely due to bulk/fragility increasing delivery risk.
5. **Payment behavior is a secondary factor.** Installments track order value sensibly but show almost no independent relationship with satisfaction.
6. **Order complexity (multi-item, multi-seller orders) and cross-state shipping are secondary/contributing drivers** — they appear to act mainly by increasing the risk of a late delivery, rather than being independent root causes.
7. **Repeat-purchase rate is low (3.1% of customers, 5.7% of revenue)** — a separate, addressable growth lever alongside the satisfaction fixes above.

**See the accompanying Analysis Report for the full business narrative and prioritized recommendations.**

## 12. Dashboard Design Updates (Power BI / HTML Dashboard)

The interactive dashboard (`Olist_PowerBI_Dashboard.html`) that accompanies this notebook and the Analysis Report was revised for visual variety and a consistent ColorBrewer-based palette. Summary of changes (full detail also lives in the Analysis Report's "Dashboard Design Updates" section):

1. **Chart type diversification** — the dashboard no longer repeats one chart type on every tab. Types were reassigned per tab based on what each metric needs to show:
   - Overview: combination bar + line chart (orders as bars, avg review score as a line).
   - Delivery & Satisfaction: ordered vertical bar chart across delay buckets.
   - Seller & Geographic Patterns: horizontal bar (late-rate ranking by state) + a pie chart (seller share by state).
   - Category Performance: horizontal bar ranking 15 categories.
   - Payment Behavior: doughnut chart (payment-method mix) + a filled line/area chart (order value across installment bins).
   - Root Cause Ranking: horizontal bar, shade intensity scaled to each driver's effect size.
2. **ColorBrewer palette** — BuPu for chart series; RdPu (`#dd3497`) for every page and card heading; PuRd (`#c994c7`) for the sidebar index/navigation. The full 9-class BuPu ramp is `#f7fcfd, #e0ecf4, #bfd3e6, #9ebcda, #8c96c6, #8c6bb1, #88419d, #810f7c, #4d004b`, but the 3 lightest stops render as near-white on the dashboard's white cards and made some bars/slices disappear. Charts now sample only the 6 visibly-contrasting stops (`#9ebcda, #8c96c6, #8c6bb1, #88419d, #810f7c, #4d004b`) through a smooth interpolation helper (`bupuShade`/`bupuSet`), so multi-category charts (the seller-concentration pie, the driver-ranking bars) still show a full light-to-dark transition without any invisible bars.
3. **Heading formatting** — every page title is centered and set in uppercase.
4. **Navigation indicator** — the sidebar index now shows a yellow dot next to whichever tab is currently open.
5. **Native Power BI (`.pbix`) file** — a `.pbix` is a compiled, proprietary binary produced by Power BI Desktop itself, so it cannot be safely hand-assembled in this environment (no Power BI Desktop, no network access) without risking a file that fails to open. The HTML dashboard is the deliverable that can be viewed directly; to rebuild it natively in Power BI Desktop, load `Raw_Data.xlsx`, use the relationships documented in the Analysis Report/Hackathon brief, and recreate the visuals listed above using the same aggregates computed in this notebook (`monthly`, `delay_bucket`, `state`, `seller_state`, `category`, `payment`, `installments`, `drivers`).